<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

**Chapter 03 &mdash; Working with Financial Data**

&copy; Dr Yves J Hilpisch | The Python Quants GmbH

http://tpq.io | [training@tpq.io](mailto:training@tpq.io) | [dyjh](http://twitter.com/dyjh)

<img src="https://hilpisch.com/pyalgo_cover_color.png" width="40%">

## Sample Data Set from Eikon

Create the local `data` folder if necessary via

    !mkdir data

In [ ]:
!mkdir data

You can register for a **free trial account for Eikon** under:

    https://refini.tv/3bvPev5

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [ ]:
pip install eikon

In [ ]:
import eikon as ek
import warnings; warnings.simplefilter('ignore')

In [ ]:
path = '/content/python_for_algo_trading_core/'  # adjust the path to your own one

In [ ]:
import configparser
config = configparser.ConfigParser()
config.read(path + 'pyalgo.cfg')

## Reading Financial Data From Different Sources

### The Data Set

In [ ]:
fn = 'AAPL.csv'

In [ ]:
with open(path+fn, 'r') as f:
    for _ in range(5):
        print(f.readline(), end='')

### Reading from a CSV File with Python

In [ ]:
import csv

In [ ]:
csv_reader = csv.reader(open(path+fn, 'r'))

In [ ]:
data = [l for l in csv_reader]

In [ ]:
data[:5]

In [ ]:
csv_reader = csv.DictReader(open(path+fn, 'r'))

In [ ]:
data = [l for l in csv_reader]

In [ ]:
data[:3]

In [ ]:
sum([float(l['CLOSE']) for l in data]) / len(data)

### Reading from a CSV File with numpy

In [ ]:
import datetime
import numpy as np

In [ ]:
# help(np.genfromtxt)

In [ ]:
t = b'2020-04-06'

In [ ]:
datetime.datetime.strptime(t.decode(), '%Y-%m-%d')

In [ ]:
con = {0: lambda t: datetime.datetime.strptime(t, '%Y-%m-%d')}

In [ ]:
sa = np.genfromtxt(path+fn, delimiter=',', dtype=None, names=True, converters=con)

In [ ]:
sa

In [ ]:
sa['CLOSE'].mean()

### Reading from a CSV File with pandas

In [ ]:
import pandas as pd
pd.__version__
import openpyxl
openpyxl.__version__

In [ ]:
data = pd.read_csv(path+fn, index_col=0,
                   parse_dates=True)

In [ ]:
data.info()

In [ ]:
data.tail()

In [ ]:
data['CLOSE'].mean()

### Exporting to Excel and JSON

In [ ]:
data.to_excel(path + 'aapl.xlsx', 'AAPL')

In [ ]:
data.to_json(path + 'aapl.json')

In [ ]:
ls -n $path/aapl.*

### Reading from Excel and JSON

In [ ]:
data_copy_1 = pd.read_excel(path + 'aapl.xlsx', 'AAPL', index_col=0)

In [ ]:
data_copy_1.head()

In [ ]:
data_copy_1.info()

In [ ]:
data_copy_2 = pd.read_json(path + 'aapl.json')

In [ ]:
data_copy_2.head()

In [ ]:
data_copy_2.info()

In [ ]:
!rm $path/aapl.*

## Working with Open Data Sources

In [ ]:
pip install quandl

In [ ]:
import quandl as q

In [ ]:
help(q.get)

In [ ]:
data = q.get('BCHAIN/MKPRU', api_key=config['quandl']['api_key'])

In [ ]:
data.info()

In [ ]:
data['Value'].resample('A').last()

In [ ]:
data = q.get('FSE/SAP_X', start_date='2018-1-1',
             end_date='2020-05-01',
             api_key=config['quandl']['api_key'])

In [ ]:
data.info()

In [ ]:
q.ApiConfig.api_key = config['quandl']['api_key']

In [ ]:
vol = q.get('VOL/MSFT')

In [ ]:
vol.iloc[:, :10].info()

In [ ]:
vol[['IvMean30', 'IvMean60', 'IvMean90']].tail()

## Refinitiv Eikon Data API

`pip install eikon` &mdash; Requires a paid subscription.

In [ ]:
import eikon as ek

In [ ]:
ek.set_app_key(config['eikon']['app_key'])

### Retrieving Historical Structured Data

In [ ]:
import warnings; warnings.simplefilter('ignore')

In [ ]:
symbols = ['AAPL.O', 'MSFT.O', 'GOOG.O']

In [ ]:
data = ek.get_timeseries(symbols, start_date='2020-01-01', end_date='2020-05-01', interval='daily', fields=['*'])

In [ ]:
# data

In [ ]:
data.keys()

In [ ]:
type(data['AAPL.O'])

In [ ]:
data['AAPL.O'].info()

In [ ]:
data['AAPL.O'].tail()

In [ ]:
%%time data = ek.get_timeseries(symbols, start_date='2020-05-05', end_date='2020-05-06', interval='minute', fields='*')

In [ ]:
print(data['GOOG.O'].loc['2020-05-05 16:00:00': '2020-05-05 16:04:00'].round(1))

In [ ]:
for sym in symbols: print('\n' + sym + '\n', data[sym].iloc[-300:-295].round(1))

In [ ]:
%%time data = ek.get_timeseries(symbols[0], start_date='2020-05-05 15:00:00', end_date='2020-05-05 15:15:00', interval='tick', fields=['*'])

In [ ]:
data.info()

In [ ]:
data.head()

In [ ]:
resampled = data.resample('30s', label='right').agg( {'VALUE': 'last', 'VOLUME': 'sum'})

In [ ]:
resampled.tail()

### Retrieving Historical Unstructured Data

In [ ]:
headlines = ek.get_news_headlines(query='R:AAPL.O macbook', count=5, date_from='2020-4-1', date_to='2020-5-1')

In [ ]:
headlines

In [ ]:
story = headlines.iloc[0]

In [ ]:
story

In [ ]:
news_text = ek.get_news_story(story['storyId'])

In [ ]:
from IPython.display import HTML

In [ ]:
HTML(news_text)

## Storing Financial Data Efficiently

### Storing DataFrame Objects

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')

In [ ]:
from sample_data import generate_sample_data

In [ ]:
print(generate_sample_data(rows=5, cols=4))

In [ ]:
%time data = generate_sample_data(rows=150, cols=10).round(4)

In [ ]:
data.plot(figsize=(10, 6), legend=False);

In [ ]:
%time data = generate_sample_data(rows=5e6, cols=10).round(4)

In [ ]:
data.info()

In [ ]:
h5 = pd.HDFStore(path + 'data.h5', 'w')

In [ ]:
%time h5['data'] = data

In [ ]:

h5

In [ ]:
ls -n /content/python_for_algo_trading_core/data.*

In [ ]:
h5.close()

In [ ]:
h5 = pd.HDFStore(path + 'data.h5', 'r')

In [ ]:
%time data_copy = h5['data']

In [ ]:
data_copy.info()

In [ ]:
h5.close()

In [ ]:
rm /content/python_for_algo_trading_core/data.h5

In [ ]:
%time data.to_hdf(path + 'data.h5', 'data', format='table')

In [ ]:
ls -n $path/data.*

In [ ]:
%time data_copy = pd.read_hdf(path + 'data.h5', 'data')

In [ ]:
data_copy.info()

In [ ]:
import tables as tb

In [ ]:
h5 = tb.open_file(path + 'data.h5', 'r')

In [ ]:
h5

In [ ]:
h5.root.data.table[:3]

In [ ]:
h5.close()

In [ ]:
!rm $path/data.h5

### Using TsTables

In [ ]:
%%time
data = generate_sample_data(rows=2.5e6, cols=5,
                            freq='1s').round(4)

In [ ]:
data.info()

You should install the `tstables` package via:

    pip install git+https://github.com/yhilpisch/tstables

In [ ]:
pip install git+https://github.com/yhilpisch/tstables

In [ ]:
import tstables

In [ ]:
import tables as tb

In [ ]:
class desc(tb.IsDescription):
    ''' Description of TsTables table structure.
    '''
    timestamp = tb.Int64Col(pos=0)
    No0 = tb.Float64Col(pos=1)
    No1 = tb.Float64Col(pos=2)
    No2 = tb.Float64Col(pos=3)
    No3 = tb.Float64Col(pos=4)
    No4 = tb.Float64Col(pos=5)

In [ ]:
h5 = tb.open_file(path + 'data.h5ts', 'w')

In [ ]:
ts = h5.create_ts('/', 'data', desc)

In [ ]:
h5

In [ ]:
%time ts.append(data)

In [ ]:
# h5

In [ ]:
import datetime

In [ ]:
start = datetime.datetime(2021, 1, 2)

In [ ]:
end = datetime.datetime(2021, 1, 3)

In [ ]:
%time subset = ts.read_range(start, end)

In [ ]:
subset.info()

In [ ]:
start = datetime.datetime(2021, 1, 2, 12, 30, 0)

In [ ]:
end = datetime.datetime(2021, 1, 5, 17, 15, 30)

In [ ]:
%time subset = ts.read_range(start, end)

In [ ]:
subset.info()

In [ ]:
h5.close()

In [ ]:
rm $path/data.h5ts

### Storing Data with SQLite3

In [ ]:
%time data = generate_sample_data(1e6, 5, '1min').round(4)

In [ ]:
data.info()

In [ ]:
import sqlite3 as sq3

In [ ]:
con = sq3.connect(path + 'data.sql')

In [ ]:
%time data.to_sql('data', con)

In [ ]:
ls -n $path/data.*

In [ ]:
query = 'SELECT * FROM data WHERE No1 > 105 and No2 < 108'

In [ ]:
%time res = con.execute(query).fetchall()

In [ ]:
res[:5]

In [ ]:
len(res)

In [ ]:
con.close()

In [ ]:
rm $path/data.*

## Oanda Data

In [ ]:
pip install git+https://github.com/yhilpisch/tpqoa

In [ ]:
import tpqoa

In [ ]:
api = tpqoa.tpqoa(path + 'pyalgo.cfg')

In [ ]:
oanda = api.get_history('SPX500_USD', '2020-05-05', '2020-05-06', 'M1', 'M')

In [ ]:
oanda.info()

In [ ]:
eikon = ek.get_timeseries('SPY', start_date='2020-05-05', end_date='2020-05-06', interval='minute', fields=['*'])

In [ ]:
eikon = eikon.loc['2020-05-05 08:00':'2020-05-05 20:00']
eikon.info()

In [ ]:
data = pd.DataFrame(eikon['CLOSE']) data['c'] = oanda['c'].shift(1) data.columns = ('eikon', 'oanda')

In [ ]:
data.dropna(inplace=True)

In [ ]:
data = data.ffill()

In [ ]:
data.head()

In [ ]:
(data / data.iloc[0]).iloc[:50].plot(figsize=(10, 6));

In [ ]:
(data / data.iloc[0]).iloc[-50:].plot(figsize=(10, 6));

In [ ]:
data['diff'] = (data / data.iloc[0])['eikon'] - (data / data.iloc[0])['oanda'] data['diff'].mean()

In [ ]:
data['diff'].hist(bins=50, figsize=(10, 6));

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>